# Part 3 — Scalable Processing
## Notebook 04. Comparable FDIC transformations in pandas, Polars and PySpark

**Research objective.** Evaluate the execution behaviour, numerical equivalence and scaling of three analytically distinct transformations on the verified FDIC bank-quarter panel. Compare pandas eager execution with Polars LazyFrame and PySpark local SQL execution. Measure from the same staged Parquet files on the Colab local disk.

**Source contract.** An existing Google Drive experiment must contain verified `acquisition`, `storage` and `quality_panel` stages. The stage is resolved by `experiment_manifest.json`, not by directory age or a hard-coded historical run ID. All earlier stages remain read-only.

**Outputs.** Controlled measured trials; full cross-engine row/value checks; six execution plans; independently verified medians; plots; source SHA-256 provenance and immutable `processing` handoff.

**Interpretation boundary.** Spark uses `local[2]` on one Colab machine, not a multi-node cluster. No future failure outcomes or retrospective institutional-exit fields are included in the transformations.


### 1. Restore the selected experiment and processing source revision

The experiment preserves the acquisition source revision and the quality-panel revision independently. If Part 3 is already published, its own source commit is restored before execution. Otherwise, the current repository commit is used and recorded as the Part 3 source revision. Dependencies are installed from pinned files; packages already satisfying the requirement are not forcibly reinstalled.


In [ ]:
from pathlib import Path
import json, re, shutil, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BFT65B_11230945')
active_path = DRIVE_ROOT / 'control' / 'active_experiment.json'
assert active_path.is_file(), 'Complete Notebooks 01–03 first'
EXPERIMENT_ID = json.loads(active_path.read_text(encoding='utf-8'))['experiment_id']
EXPERIMENT_DIR = DRIVE_ROOT / 'experiments' / EXPERIMENT_ID
PART1_COMMIT = json.loads((EXPERIMENT_DIR/'experiment_config.lock.json').read_text())['git_commit']
QUALITY_MANIFEST = json.loads((EXPERIMENT_DIR/'quality_panel'/'part2_manifest.json').read_text())
PART2_COMMIT = QUALITY_MANIFEST['code_commit']

REPO_DIR = Path('/content/bft65b_code')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--quiet',
    'https://github.com/NguyenDuong58/11230945-NguyenBaDuong.git',str(REPO_DIR)], check=True)
part3_manifest = EXPERIMENT_DIR/'processing'/'part3_manifest.json'
if part3_manifest.is_file():
    PART3_COMMIT = json.loads(part3_manifest.read_text())['code_commit']
    assert re.fullmatch(r'[0-9a-f]{40}', PART3_COMMIT)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','--quiet',PART3_COMMIT],check=True)
else:
    PART3_COMMIT = subprocess.check_output(
        ['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
assert subprocess.check_output(
    ['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip() == PART3_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','-r',
                str(REPO_DIR/'requirements_part3.txt')],check=True)
# Avoid stale modules if a previous notebook ran in this same Python runtime.
for name in list(sys.modules):
    if name == 'fdic_pipeline' or name.startswith('fdic_pipeline.'):
        del sys.modules[name]
sys.path.insert(0,str(REPO_DIR/'src'))
print('Experiment:',EXPERIMENT_ID)
print('Acquisition commit:',PART1_COMMIT)
print('Quality-panel commit:',PART2_COMMIT)
print('Processing commit:',PART3_COMMIT)


### 2. Verify installed engines and run the complete test suite

The verification gate includes all Parts 1–3 tests and Parquet integration checks that execute pandas, Polars and PySpark on identical synthetic reporting histories. The notebook stops if any test fails, errors, or is skipped. Spark 4.0.1 requires Java 17 or later.


In [ ]:
import importlib.metadata
import xml.etree.ElementTree as ET

for package in ('pandas','polars','pyspark','pyarrow','psutil'):
    print(package, importlib.metadata.version(package))
java_version = subprocess.run(['java','-version'],capture_output=True,text=True,check=True).stderr
print(java_version.splitlines()[0])
assert re.search(r'version "(?:17|18|19|2[0-9])',java_version), 'Java 17+ is required'
TEST_XML = Path('/content/bft65b_part3_tests.xml')
subprocess.run([sys.executable,'-m','pytest','-q','-rs','--junitxml',str(TEST_XML),
                str(REPO_DIR/'tests')],cwd=REPO_DIR,check=True)
suites = ET.parse(TEST_XML).getroot().findall('.//testsuite')
assert suites, 'No test suites were recorded'
result={key:sum(int(item.get(key,0)) for item in suites)
        for key in ('tests','failures','errors','skipped')}
assert result['failures']==result['errors']==result['skipped']==0, result
print('Full Parts 1–3 test gate: PASSED',result)


### 3. Verify acquisition, storage and quality-panel lineage

The controller rejects a changed parent validation hash, missing quarter, changed source SHA-256, incompatible schema, or retrospective target field. The 48-column clean panel is the only computational input; failure-linkage and terminal-exit audit tables are excluded.


In [ ]:
from fdic_pipeline.project_flow import get_active, resolve_stage, stage_dir
from fdic_pipeline.scalable_processing import input_contract
ACTIVE=get_active(DRIVE_ROOT)
assert ACTIVE['experiment_id']==EXPERIMENT_ID
for stage in ('acquisition','storage','quality_panel'):
    assert resolve_stage(DRIVE_ROOT,stage)['status']=='PASSED'
SNAP=input_contract(DRIVE_ROOT,check_data_hashes=True)
assert SNAP['panel_fields']==48
assert len(SNAP['partitions'])==170
assert SNAP['part2_commit']==PART2_COMMIT
assert SNAP['samples']['full_170']['rows']==SNAP['panel_rows']
assert SNAP['samples']['recent_85']['quarters']==85
print('Source input contract: PASSED')
print('Panel rows:',SNAP['panel_rows'])
print('Schema fields:',SNAP['panel_fields'])
print('Sample definitions:',SNAP['samples'])
print('Quality-stage validation SHA-256:',SNAP['quality_verification_sha256'])


### 4. Prespecify analytical workloads, engines and measurement rules

**W1 — Quarter aggregation.** Aggregate the six selected source fields for observations from 2000Q1: bank-quarter count, total assets, total deposits, mean ROA and the count of extreme-ROE flags.

**W2 — Adjacent-quarter asset growth.** Sort observations by CERT and REPDTE; compute lagged assets within the same CERT; calculate percentage growth **only when the prior observed report is exactly one quarter earlier**; aggregate the number of valid adjacent pairs and their mean growth by reporting date.

**W3 — State-year risk aggregation.** Group by reporting year and source state abbreviation; aggregate assets, gross loans, mean ROA, negative-equity indicators, loan-exposure flags and the number of reports with any missing financial field. Source-null state labels are grouped under `MISSING`.

**Scales.** The latest 85 reporting quarters and all 170 reporting quarters; neither synthetic replication nor sampling with replacement is used. As the institution population changes over time, size effects are observational, not a pure controlled input-row elasticity.

**Timing.** One warmup and five measured repetitions per engine–workload–scale combination, seeded randomized order and fresh worker processes. Engine initialization is excluded; local Parquet reads, transformations and result materialization are timed. Process-tree RSS is sampled at approximately 50 ms intervals, including the Spark JVM. All engines read the same `/content` Parquet snapshot and use the same projection and filter definitions.


In [ ]:
from fdic_pipeline.processing_engines import ENGINE_NAMES,WORKLOADS,PARQUET_COLS
from fdic_pipeline.scalable_processing import DEFAULT_REPETITIONS,DEFAULT_WARMUPS,SAMPLE_NAMES
REPETITIONS=DEFAULT_REPETITIONS
WARMUPS=DEFAULT_WARMUPS
SEED=11230945
assert len(WORKLOADS)==3 and len(ENGINE_NAMES)==3 and len(SAMPLE_NAMES)==2
assert REPETITIONS>=3 and WARMUPS>=1
print('Engines:',ENGINE_NAMES)
print('Workloads:',WORKLOADS)
for w in WORKLOADS:
    print(w,'requires',PARQUET_COLS[w])
print('Expected measured trials:',len(ENGINE_NAMES)*len(WORKLOADS)*len(SAMPLE_NAMES)*REPETITIONS)
print('Expected warmups:',len(ENGINE_NAMES)*len(WORKLOADS)*len(SAMPLE_NAMES)*WARMUPS)
print('Expected cross-engine parity comparisons:',2*len(WORKLOADS)*len(SAMPLE_NAMES)*(WARMUPS+REPETITIONS))


### 5. Run the controlled benchmark and publish the processing stage

The driver copies checksum-validated panel partitions to local scratch and creates an 85-quarter subset using hard links where available. Each trial runs in a separate Python process. A full pandas–Polars and pandas–PySpark value-level comparison is required for each workload, scale and repetition, including warmups. No stage is published when comparisons disagree.

Results are written under `experiments/<experiment_id>/processing/` and sealed by SHA-256 after independent evidence verification. If the run is interrupted before publication, rerunning this section rebuilds only the incomplete processing stage; acquisition, storage and quality-panel outputs remain untouched.


In [ ]:
from fdic_pipeline.project_flow import HandoffError
PROCESSING_DIR=stage_dir(DRIVE_ROOT,'processing')
SCRATCH=Path('/content')/('bft65b_p3_'+EXPERIMENT_ID)
LOG_FILE=EXPERIMENT_DIR/'run_logs'/'part3_scalable_processing.log'
try:
    PREVIOUS=resolve_stage(DRIVE_ROOT,'processing')
except HandoffError as exc:
    if 'has not passed verification' not in str(exc):
        raise
    PREVIOUS=None
if PREVIOUS is None:
    free_gib=shutil.disk_usage('/content').free/(2**30)
    assert free_gib >= 3.0, f'At least 3 GiB local disk space required: {free_gib:.2f} GiB available'
    command=[sys.executable,'-u',str(REPO_DIR/'run_scalable_processing.py'),'build',
             '--drive-root',str(DRIVE_ROOT),'--code-commit',PART3_COMMIT,
             '--scratch',str(SCRATCH),'--repetitions',str(REPETITIONS),
             '--warmups',str(WARMUPS),'--replace-incomplete','--log-file',str(LOG_FILE)]
    subprocess.run(command,cwd=REPO_DIR,check=True)
else:
    print('Processing stage already published; running read-only verification')
print('Processing evidence directory:',PROCESSING_DIR)
print('Persistent log:',LOG_FILE)


### 6. Inspect the measured engine comparison and sample scaling

Median times are calculated independently from recorded trials. Speedup is defined as pandas median elapsed time divided by the comparison engine median for the same workload and sample. A value above one indicates a faster comparison engine. The full/recent scaling ratio is not a fixed-row growth law because institution counts differ across periods.


In [ ]:
import csv
from IPython.display import display, Markdown

def show_csv(title, rel, limit=25):
    with (PROCESSING_DIR/rel).open(newline='',encoding='utf-8') as f:
        rows=list(csv.DictReader(f))
    display(Markdown('**'+title+'**'))
    assert rows, f'Empty evidence file: {rel}'
    fields=list(rows[0])
    rows=rows[:limit]
    display(Markdown(' | '.join(fields)+'\n'+' | '.join(['---']*len(fields))+'\n'+
      '\n'.join(' | '.join(str(row[k]) for k in fields) for row in rows)))

show_csv('Measured run medians and speedups','performance_summary.csv',18)
show_csv('Observed full/recent scaling','scaling_summary.csv',9)
show_csv('Cross-engine equivalence checks (first 12)','equivalence.csv',12)


### 7. Inspect optimized Polars and Spark plans

The Polars plans are generated with `LazyFrame.explain(optimized=True)`; the Spark plans contain analyzed, optimized and physical plan information from query execution. Look for Parquet scans, projection, filter pushdown, aggregation, sorting, window operations and Spark exchanges. Plan-inspection indicators are recorded as string matches and must not be presented as proof that a particular optimization occurred without reading the plan.


In [ ]:
show_csv('Execution-plan inspection','plan_inspection.csv',6)
for path in ('plans/polars_adjacent_asset_growth.txt',
             'plans/pyspark_adjacent_asset_growth.txt'):
    content=(PROCESSING_DIR/path).read_text(encoding='utf-8')
    display(Markdown('**'+path+' — first 35 lines**'))
    display(Markdown('```text\n'+'\n'.join(content.splitlines()[:35])+'\n```'))
print('All 6 complete plan files and the interpretation guide are preserved on Drive')


### 8. Display controlled benchmark figures

Figures are generated only from measured trials and are stored with SHA-256 evidence. Runtime plots should be interpreted alongside the exact workload, sample, CPU environment, and process memory protocol.


In [ ]:
from IPython.display import Image
for name in ('figures/median_runtime.png',
             'figures/scaling.png','figures/median_process_rss.png'):
    display(Markdown('**'+Path(name).stem.replace('_',' ').title()+'**'))
    display(Image(filename=str(PROCESSING_DIR/name),width=850))


### 9. Independently verify the published Part 3 evidence

The verification step rechecks all upstream stage references, SHA-256 of the 170 source partitions, every evidence file and execution plan, the completeness of each measurement round, the number of numerical comparisons and the median/ratio arithmetic. The stage and its parent-validation references are immutable after publication.


In [ ]:
from fdic_pipeline.scalable_processing import verify_project_stage
VERIFIED=verify_project_stage(DRIVE_ROOT)
assert VERIFIED['success'] is True
assert VERIFIED['source_quarters']==170
assert VERIFIED['panel_rows']==SNAP['panel_rows']
assert VERIFIED['measured_trials']==90
assert VERIFIED['warmup_trials']==18
assert VERIFIED['numerical_comparisons']==72
assert VERIFIED['plan_files']==6
assert VERIFIED['cluster_tested'] is False
print('Part 3: PASSED')
for key in ('panel_rows','measured_trials','warmup_trials',
            'numerical_comparisons','plan_files','evidence_artifacts_verified'):
    print(key+':',VERIFIED[key])
print('Verified source-to-output lineage: PASSED')


### 10. Establish the processing evidence contract for Parts 4–6

Part 4 uses the verified `quality_panel/panel/` as the input for point-in-time feature engineering. It may reference the `processing` stage for measurement provenance, but must not use computed aggregate summaries as contemporaneous bank-level predictors, nor read retrospective failure/exit audit fields as predictors. Part 5 will operate on separately constructed labels and modelling samples.


In [ ]:
STAGE=resolve_stage(DRIVE_ROOT,'processing')
assert STAGE['status']=='PASSED'
assert STAGE['validation']['part3_manifest_sha256']==VERIFIED['part3_manifest_sha256']
assert STAGE['parent_verification_sha256']==SNAP['quality_verification_sha256']
print('Part 4 panel source:',Path(resolve_stage(DRIVE_ROOT,'quality_panel')['path'])/'panel')
print('Part 3 evidence:',PROCESSING_DIR)
print('Processing code commit:',VERIFIED['code_commit'])
print('Handoff state: PASSED')
